# Caching functions that read and write files

`fleche` is a content-addressed cache.  When a cached function takes or returns a
`pathlib.Path`, fleche stores the file's (or directory tree's) **contents** — not
just the path string — so results are portable and reproducible across machines.

A path is keyed by its content, so:

- two files with identical bytes share one storage entry (deduplication), and
- re-calling a function with the same file content is a cache hit, even if the
  file lives at a different location.

The in-memory cache (`cache("memory")`) and every default value storage now carry
this behaviour out of the box — no custom storage composition required.

In [1]:
import tempfile
from pathlib import Path
from subprocess import run

import fleche as fl
from fleche import fleche

fl.cache("memory")          # activate a transient in-memory cache
c = fl.cache()              # grab the active cache to introspect later

# The default value storage stores Paths by content:
[k.__name__ for k in type(c.values).__mro__ if k.__name__.endswith("Mixin")]

['PerKeyLockMixin', 'DestructuringMixin', 'PathValueMixin', 'ValueMixin']

In [2]:
# A scratch directory for the files our functions produce.
WORK = Path(tempfile.mkdtemp(suffix="-fleche-files"))
WORK

PosixPath('/tmp/tmpyxng_i3t-fleche-files')

## Producing a file

A cached function can create a file and return its `Path`.  fleche stores the
bytes; on a cache hit it hands back a freshly materialized temporary `Path` with
the same contents.  The `print` fires only when the body actually runs.

In [3]:
@fleche
def write(text, repeat=1, name="out.txt"):
    print("  [write] running:", repr(text))
    f = WORK / name
    f.write_text(text * repeat)
    return f

f = write("hello", 2)
print("returned:", type(f).__name__, "->", f.read_text())

  [write] running: 'hello'
returned: PosixPath -> hellohello


In [4]:
# Same arguments -> cache hit -> the body does NOT run (no "[write] running").
again = write("hello", 2)
print("from cache:", again.read_text())

from cache: hellohello


## Consuming a file

A function can take a `Path` argument; fleche keys the call on the file's
content.  Feeding it a path produced by another cached function chains the two.

In [5]:
@fleche
def wordcount(path: Path):
    print("  [wordcount] running:", path.name)
    return len(path.read_text().split())

print("count:", wordcount(write("a quick brown fox", 1, name="sentence.txt")))
# Re-run: both `write` and `wordcount` are served from cache.
print("count again:", wordcount(write("a quick brown fox", 1, name="sentence.txt")))

  [write] running: 'a quick brown fox'
  [wordcount] running: sentence.txt
count: 4
count again: 4


## Directories

Returning a directory `Path` stores the whole tree.  On load it is rebuilt under
a temporary directory, so `iterdir()` / `rglob()` work exactly as before.

In [6]:
@fleche
def make_tree(seed):
    print("  [make_tree] running:", seed)
    d = WORK / f"tree-{seed}"
    d.mkdir(exist_ok=True)
    (d / "top.txt").write_text(seed)
    (d / "sub").mkdir(exist_ok=True)
    (d / "sub" / "leaf.bin").write_bytes(seed.encode() * 3)
    return d

@fleche
def total_bytes(d: Path):
    print("  [total_bytes] running:", d.name)
    return sum(p.stat().st_size for p in d.rglob("*") if p.is_file())

tree = make_tree("alpha")
sorted(p.relative_to(tree).as_posix() for p in tree.rglob("*"))

  [make_tree] running: alpha


['sub', 'sub/leaf.bin', 'top.txt']

In [7]:
# End-to-end cache hit: make_tree("alpha") and total_bytes both come from cache.
total_bytes(make_tree("alpha"))

  [total_bytes] running: ff88b2cec3e828c231a8a9df977be4f4cfa2e938cc0cdc41aac1e03350f5af8b


20

## Content-addressing

Files are stored under the digest of their contents, so identical bodies are
stored once regardless of filename or which call produced them.

In [8]:
# Two calls, different names, identical body -> the content is stored once.
write("shared body", 1, name="left.txt")
write("shared body", 1, name="right.txt")

body_key = fl.digest.digest(b"shared body")     # content blobs are plain bytes
print("shared body stored once:", body_key in set(c.values.list()))

  [write] running: 'shared body'
  [write] running: 'shared body'
shared body stored once: True


In [9]:
# What a `Path` actually becomes in storage: its content as plain `bytes` under
# its own digest, plus a small record pairing that content with a name.  Note
# that left.txt and right.txt reference the *same* content digest.
for blob in c.values.storage.values():
    if type(blob).__name__ in ("FileBlob", "DirectoryBlob"):
        print(blob)


FileBlob('out.txt', '785d68f8426805e292630852bdedb46dd56ac44dcb7047740d30704f3d84d4fa')
FileBlob('sentence.txt', '0f61b76af53fa2dc41528c3866206d22ceb3b7f560e7f42aacc006fc5ace228c')
DirectoryBlob({'leaf.bin': 'c70f6db1a5371bc6046fb5a040fd13bd5220c78908eac8126c1361daad854904'})
DirectoryBlob({'sub': '7aa02a10cd58fe7b0f16ef0e06b255d7842ce362687e489e941281d27761d95a', 'top.txt': '8eb42147b1727df4b082ebc0bdfc5fbaea064308411a4d801cae67c908ce4287'})
FileBlob('left.txt', '32cbd77d1dbff488cd42dc84ea72ebd47358fbf321412b35a5c1084e36f5b775')
FileBlob('right.txt', '32cbd77d1dbff488cd42dc84ea72ebd47358fbf321412b35a5c1084e36f5b775')


## A real workflow: orchestrating shell scripts

The motivating use case: wrap command-line tools that read and write files.  Each
step runs in a working directory, produces files, and the whole pipeline is
cached by content.

Each step returns the `subprocess.CompletedProcess` that `run()` produced.  fleche
digests those directly — by `args`, `returncode`, `stdout`, and `stderr` — so the
captured output participates in the key with no extra setup.  For a type fleche
does not know, `fl.digest.add_hook((TheType, fn))` is the extension point.

In [10]:
@fleche
def shell(cwd, prog, args=(), stdin=b""):
    print("  [shell] running:", prog, *args)
    ret = run([prog, *args], cwd=cwd, capture_output=True, input=stdin)
    return cwd, ret

@fleche
def pipeline(content):
    print("  [pipeline] running:", content)
    work = Path(tempfile.mkdtemp(suffix="-pipeline"))
    (work / "input.txt").write_bytes(content)
    shell(work, "cp", ["input.txt", "copy.txt"])                       # produce a file
    _, upper = shell(work, "tr", ["a-z", "A-Z"], stdin=content)        # capture stdout
    (work / "shout.txt").write_bytes(upper.stdout)
    return work

In [11]:
print("--- first run ---")
out = pipeline(b"hello world")
print("produced:", sorted(p.name for p in out.iterdir()))
print("shout.txt:", (out / "shout.txt").read_text())

--- first run ---
  [pipeline] running: b'hello world'
  [shell] running: cp input.txt copy.txt
  [shell] running: tr a-z A-Z
produced: ['copy.txt', 'input.txt', 'shout.txt']
shout.txt: HELLO WORLD


In [12]:
print("--- second run: fully cached (no body / shell prints) ---")
out2 = pipeline(b"hello world")
print("same files:", sorted(p.name for p in out2.iterdir()))

--- second run: fully cached (no body / shell prints) ---
same files: ['copy.txt', 'input.txt', 'shout.txt']


In [13]:
# Every shell invocation fleche recorded:
shell.query().table()

,name,module,timestart,timestop,walltime
6f89,shell,__main__,2026-08-06 19:58:49.251591921+00:00,2026-08-06 19:58:49.259691+00:00,0.008099
a668,shell,__main__,2026-08-06 19:58:49.261044025+00:00,2026-08-06 19:58:49.264387608+00:00,0.003344
